# Cellpose-SAM

Fine-tunes the ImageGrains 2.0 model on the tile archive, one run per fold.

Cellpose-SAM is locked to 256 px crops, so the crops come out of the archive at
384 and cellpose takes its 256 from inside them. The margin means a rotation
never pulls in empty corners.

Crops only come from positions where the whole 384 window is inspected ground,
so the label raster is the full truth for every pixel the net sees. Hillside and
anything past the image edge is never used.

Measured on this GPU: 0.57 s per crop at batch 1, 8.5 GB peak. Batch 2 fits in
13.2 GB, batch 4 does not.

Results go to `results/cellpose_sam_<fold>.json` after every fold, so stopping
half way still leaves usable numbers.

In [ ]:
import sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from cellpose import models, train

warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
import config
from common import tiles, crops, evaluate, folds

FOLDS = ["fold1"]            # ["fold1", "fold2", "fold3", "fold4"] for the full run
EPOCHS = 100
PER_EPOCH = 250              # crops per epoch, drives the runtime
BATCH = 1
LR = 1e-5
WEIGHT_DECAY = 0.1
SCALE_RANGE = 0.5            # 0.75 to 1.25
SAVE_EVERY = 20
BASE = config.MODELS / "IG2_full_set_cp_SAM"

print("gpu:", torch.cuda.get_device_name(0))
print("base model:", BASE.name, "-", BASE.exists())
print(f"{EPOCHS} x {PER_EPOCH} = {EPOCHS * PER_EPOCH} crops per fold, "
      f"about {EPOCHS * PER_EPOCH * 0.57 / 3600:.1f} h")

## Tiles per fold

Validation is one fold, training is everything else except the test block. The
test block stays untouched until the very end.

In [ ]:
idx = pd.DataFrame(tiles.read_index())
print(idx.groupby("fold").agg(tiles=("name", "size"), boulders=("n_inner", "sum")))
print(f"\ntest block held back: {int((idx.fold == 'test').sum())} tiles")

## Crops

One crop centred on each boulder plus 30 % random corridor positions. Tiles
without labels are left out, their background is already in every other crop.

In [ ]:
def crop_sets(fold):
    train_names = idx.loc[idx.fold.isin(folds.training_folds(fold))
                          & (idx.n_inner > 0), "name"].tolist()
    val_names = idx.loc[(idx.fold == fold) & (idx.n_inner > 0), "name"].tolist()

    print(f"{fold}: {len(train_names)} training tiles, {len(val_names)} validation tiles")
    xtr, ytr = crops.build(train_names, seed=0, verbose=False)
    xva, yva = crops.build(val_names, seed=1, verbose=False)
    print(f"  {len(xtr)} training crops, {len(xva)} validation crops")
    return xtr, ytr, xva, yva, val_names

## What the crops look like

In [ ]:
xtr, ytr, xva, yva, val_names = crop_sets(FOLDS[0])

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (a, b) in zip(axes.ravel(), zip(xtr[:8], ytr[:8])):
    lo, hi = np.percentile(a[a > 0], config.PERCENTILE)
    ax.imshow(np.clip((a - lo) / max(hi - lo, 1), 0, 1), cmap="gray")
    ax.contour(b > 0, levels=[0.5], colors="red", linewidths=0.8)
    ax.set_title(f"{int(b.max())} boulders", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## Train

Checkpoints every 20 epochs. Cellpose has no early stopping, so all of them get
scored afterwards and the best one wins.

In [ ]:
def run(fold, xtr, ytr, xva, yva):
    out = config.MODELS / f"cellpose_sam_{fold}"
    out.mkdir(parents=True, exist_ok=True)

    net = models.CellposeModel(gpu=True, pretrained_model=str(BASE)).net
    t0 = time.time()
    path, train_loss, test_loss = train.train_seg(
        net,
        train_data=xtr, train_labels=ytr,
        test_data=xva, test_labels=yva,
        batch_size=BATCH, n_epochs=EPOCHS, nimg_per_epoch=PER_EPOCH,
        learning_rate=LR, weight_decay=WEIGHT_DECAY,
        scale_range=SCALE_RANGE, bsize=256,
        normalize={"percentile": list(config.PERCENTILE)},
        min_train_masks=1,
        save_path=str(out), save_every=SAVE_EVERY, save_each=True,
        model_name=fold)
    print(f"\n{(time.time() - t0) / 3600:.2f} h")
    return path, train_loss, test_loss, out

In [ ]:
path, train_loss, test_loss, out_dir = run(FOLDS[0], xtr, ytr, xva, yva)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(train_loss, label="train", lw=1)
ax.plot(np.where(test_loss > 0, test_loss, np.nan), label="validation", lw=1.4)
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()
plt.tight_layout()
plt.show()

## Score the checkpoints

On whole validation tiles, not crops, because that is how it gets applied.
Only objects whose centre falls inside the corridor and inside the tile's own
inner region are counted, so nothing is scored twice and nothing is scored on
ground that was never inspected.

In [ ]:
def predict_tile(model, name):
    img, true, valid = crops.read_tile(name)
    pred, _, _ = model.eval(img.astype(np.float32), batch_size=8,
                            normalize={"percentile": list(config.PERCENTILE)},
                            flow_threshold=0.4, cellprob_threshold=0.0,
                            min_size=15)
    keep = valid.astype(bool) & evaluate.inner_mask(img.shape)
    return evaluate.restrict(pred.astype(np.uint16), keep), evaluate.restrict(true, keep)


def score_checkpoint(ckpt, names):
    model = models.CellposeModel(gpu=True, pretrained_model=str(ckpt))
    rows = []
    for n in names:
        p, t = predict_tile(model, n)
        rows.append(evaluate.score(p, t))
    del model
    torch.cuda.empty_cache()
    return evaluate.summarise(rows)

In [ ]:
ckpts = sorted((out_dir / "models").glob(f"{FOLDS[0]}*"))
print(f"{len(ckpts)} checkpoints")

rows = []
for c in ckpts:
    s = score_checkpoint(c, val_names)
    s["checkpoint"] = c.name
    rows.append(s)
    print(f"  {c.name:28} AP {s['ap']:.3f}  P {s['precision']:.3f}  "
          f"R {s['recall']:.3f}  IoU {s['mean_iou']:.3f}")

curve = pd.DataFrame(rows).set_index("checkpoint")
best = curve.ap.idxmax()
print(f"\nbest: {best}, AP {curve.ap.max():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(curve.ap.values, "o-", label="AP")
axes[0].plot(curve.precision.values, "o-", lw=1, label="precision")
axes[0].plot(curve.recall.values, "o-", lw=1, label="recall")
axes[0].axhline(0.16, color="k", ls="--", lw=1)
axes[0].set_xlabel("checkpoint")
axes[0].legend()
axes[0].set_title("dashed line is OrthoSAM")

axes[1].plot(curve.d50_pred.values, "o-", label="D50 predicted")
axes[1].plot(curve.d50_true.values, "k--", lw=1, label="D50 mapped")
axes[1].plot(curve.d84_pred.values, "o-", label="D84 predicted")
axes[1].plot(curve.d84_true.values, "k:", lw=1, label="D84 mapped")
axes[1].set_xlabel("checkpoint")
axes[1].set_ylabel("m")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Predictions against the hand drawn outlines

In [ ]:
model = models.CellposeModel(gpu=True, pretrained_model=str(out_dir / "models" / best))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, name in zip(axes.ravel(), val_names[:6]):
    img, _, _ = crops.read_tile(name)
    p, t = predict_tile(model, name)
    lo, hi = np.percentile(img[img > 0], config.PERCENTILE)
    ax.imshow(np.clip((img - lo) / max(hi - lo, 1), 0, 1), cmap="gray")
    ax.contour(t > 0, levels=[0.5], colors="red", linewidths=0.8)
    ax.contour(p > 0, levels=[0.5], colors="lime", linewidths=0.8)
    ax.set_title(f"{name}  mapped {int(t.max())}  found {int(p.max())}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## Save

In [ ]:
result = dict(model="cellpose_sam", fold=FOLDS[0], base=BASE.name,
              epochs=EPOCHS, per_epoch=PER_EPOCH, batch=BATCH,
              lr=LR, scale_range=SCALE_RANGE, res=config.RES,
              crop=crops.SIZE, bsize=256,
              n_train_crops=len(xtr), n_val_crops=len(xva),
              n_val_tiles=len(val_names),
              best_checkpoint=best,
              curve=curve.reset_index().to_dict("records"),
              **final)

config.RESULTS.mkdir(parents=True, exist_ok=True)
f = config.RESULTS / f"cellpose_sam_{FOLDS[0]}.json"
f.write_text(json.dumps(result, indent=1), encoding="utf-8")
print(f)
print({k: round(v, 3) for k, v in result.items() if isinstance(v, float)})

## The other folds

Set `FOLDS` at the top to all four and rerun from the crops cell, or leave it
running headless:

```bash
screen -S cpsam
jupyter nbconvert --to notebook --execute --inplace 02_cellpose_sam.ipynb
```